# Web Skin 마지막 Validation 실험: PMG · EfficientNet-B1 · 384

현재 Validation 선두인 `PMG·B0·256·CE`를 다시 학습하지 않고 저장된 기준값으로 사용합니다.
새로 학습하는 모델은 **PMG·B1·384·CE 하나뿐**입니다. 동일한 데이터와 클래스 순서를
사용하며 최종 후보를 고르기 전이므로 Test는 열지 않습니다.

- PMG: https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123650154.pdf
- EfficientNet: https://proceedings.mlr.press/v97/tan19a.html


## 1. 이번 실험의 한 가지 가설

PMG의 `8×8 → 4×4 → 2×2 → 원본` 학습 방식, CE Loss, seed와 학습 횟수는 유지합니다.
Hair에서 가장 좋았던 B1·384 규모를 적용했을 때 얼굴 피부의 작은 차이를 더 잘 구분하는지
검증합니다.

기존 PMG는 batch 32였지만 B1·384의 GPU 메모리를 위해 이번 실행은 batch 16을 사용합니다.
이는 필요한 동반 변경으로 결과에 기록됩니다. 실제 사용 시에는 384×384 얼굴 사진 한 장을
입력하고, 세 branch와 fusion logit을 합산합니다.


In [ ]:
DOMAIN = 'web_skin'
PROJECT_ROOT = '/content/drive/MyDrive/mediflow_Project'
DATA_ZIP = '/content/drive/MyDrive/mediflow_Project/datasets/web_skin_datasets.zip'
AUDIT_DIR = ''
EXPECTED_DATA_SHA256 = 'f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d'
RESUME_DIR = ''
MODE = 'web_skin_pmg_b1_384'
SEED = 42
SEEDS = [42]
BATCH_SIZE = 16
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 10
EXTENSION_EPOCHS = 1  # 사용하지 않음
TRAIN_VARIANT = 'augmented'
RUN_EXPERIMENTS = ['pmg_b1_384_ce_seed_42']
PMG_JIGSAW_GRIDS = [8, 4, 2]


## 2. Drive·GPU·데이터 확인

Drive를 새로 연결하고 `web_skin_datasets.zip`이 실제 ZIP인지 확인합니다. 384 입력은 계산량이
크므로 Colab 런타임에서 GPU를 선택해야 합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount('/content/drive', force_remount=True)

data_path = Path(DATA_ZIP)
dataset_dir = Path(PROJECT_ROOT) / 'datasets'
if not data_path.is_file():
    available = sorted(path.name for path in dataset_dir.iterdir()) if dataset_dir.is_dir() else []
    raise FileNotFoundError(
        f'Web Skin 데이터 파일을 찾을 수 없습니다: {data_path}\n'
        f'datasets 폴더에서 확인된 항목: {available}'
    )
if not zipfile.is_zipfile(data_path):
    raise ValueError(f'ZIP 형식으로 열 수 없습니다: {data_path}')
print('데이터 파일 확인:', data_path)
print('데이터 파일 크기:', round(data_path.stat().st_size / 1024**3, 2), 'GiB')

%pip -q install tensorflow==2.20.0 keras==3.13.2 pandas matplotlib pillow tqdm
import tensorflow as tf
import keras
if tf.__version__ != '2.20.0' or keras.__version__ != '3.13.2':
    raise RuntimeError('Colab 세션을 다시 시작한 뒤 처음부터 실행하세요.')
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('런타임 유형에서 GPU를 선택하세요.')
print('GPU:', tf.config.list_physical_devices('GPU'))


## 3. 내장 재현 코드

별도 Python 파일은 필요 없습니다. 실행 코드와 해시는 결과 폴더에 자동 저장됩니다. 이 셀은
수정하지 않습니다.


In [ ]:
import sys, types
SOURCES = {'common_engine': '"""Sequential domain-configured experiments; validation selection precedes any test inference.\n\nThe Colab notebook embeds an exact copy of this module so no repository checkout\nis needed in Colab. Completed trials are reused only after artifact verification.\nInterrupted attempts are preserved and restarted, not resumed mid-epoch.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nPROTOCOL = "mediflow_common_v1"\nTRIALS = [\n    {"id": "b0_224_ce", "backbone": "B0", "size": 224, "loss": "ce"},\n    {"id": "b0_256_ce", "backbone": "B0", "size": 256, "loss": "ce"},\n    {"id": "b0_256_ls005", "backbone": "B0", "size": 256, "loss": "ls005"},\n    {"id": "b0_256_focal15", "backbone": "B0", "size": 256, "loss": "focal15"},\n    {"id": "b1_256_ls005", "backbone": "B1", "size": 256, "loss": "ls005"},\n]\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(".json-" + uuid.uuid4().hex[:12] + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef classification_metrics(truth, probabilities, count=5):\n    truth = np.asarray(truth, dtype=np.int64)\n    probabilities = np.asarray(probabilities)\n    if (\n        truth.ndim != 1\n        or not len(truth)\n        or probabilities.shape != (len(truth), count)\n        or not np.isfinite(probabilities).all()\n        or np.any(truth < 0)\n        or np.any(truth >= count)\n    ):\n        raise ValueError("Invalid evaluation arrays")\n    predictions = probabilities.argmax(axis=1)\n    cm = np.bincount(count * truth + predictions, minlength=count * count).reshape(count, count)\n    tp = np.diag(cm).astype(float)\n    precision = np.divide(tp, cm.sum(0), out=np.zeros(count), where=cm.sum(0) != 0)\n    recall = np.divide(tp, cm.sum(1), out=np.zeros(count), where=cm.sum(1) != 0)\n    f1 = np.divide(\n        2 * precision * recall,\n        precision + recall,\n        out=np.zeros(count),\n        where=precision + recall != 0,\n    )\n    return {\n        "accuracy": float(np.mean(truth == predictions)),\n        "macro_f1": float(f1.mean()),\n        "class_f1": f1.tolist(),\n        "precision": precision.tolist(),\n        "recall": recall.tolist(),\n        "support": cm.sum(1).tolist(),\n        "confusion_matrix": cm.tolist(),\n        "count": len(truth),\n    }\n\n\ndef predict_dataset(model, dataset):\n    truth, probabilities = [], []\n    for images, labels in dataset:\n        probabilities.extend(model(images, training=False).numpy())\n        truth.extend(np.argmax(labels.numpy(), axis=1))\n    return np.asarray(truth, dtype=np.int64), np.asarray(probabilities)\n\n\ndef save_predictions(path, paths, truth, probabilities):\n    if len(paths) != len(truth):\n        raise ValueError("File order and prediction count differ")\n    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "path",\n                "true_index",\n                "pred_index",\n                *[f"prob_C{i}" for i in range(probabilities.shape[1])],\n            ]\n        )\n        for name, target, probs in zip(paths, truth, probabilities, strict=True):\n            writer.writerow([name, int(target), int(probs.argmax()), *map(float, probs)])\n\n\ndef loss_function(name):\n    if name == "ce":\n        return keras.losses.CategoricalCrossentropy()\n    if name == "ls005":\n        return keras.losses.CategoricalCrossentropy(label_smoothing=0.05)\n    if name == "focal15":\n        return keras.losses.CategoricalFocalCrossentropy(alpha=1.0, gamma=1.5)\n    raise ValueError(name)\n\n\ndef build_model(spec):\n    builder = {\n        "B0": keras.applications.EfficientNetB0,\n        "B1": keras.applications.EfficientNetB1,\n        "V2S": keras.applications.EfficientNetV2S,\n    }\n    size = spec["size"]\n    backbone = builder[spec["backbone"]](\n        include_top=False, weights="imagenet", input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = backbone(inputs, training=False)\n    features = keras.layers.GlobalAveragePooling2D()(features)\n    features = keras.layers.Dropout(0.3)(features)\n    outputs = keras.layers.Dense(spec["class_count"], activation="softmax")(features)\n    model = keras.Model(inputs, outputs)\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-4),\n        loss=loss_function(spec["loss"]),\n        metrics=["accuracy"],\n    )\n    return model\n\n\ndef configure_partial(model, loss_name):\n    backbones = [\n        layer\n        for layer in model.layers\n        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()\n    ]\n    if len(backbones) != 1:\n        raise ValueError("Expected one EfficientNet backbone")\n    backbone = backbones[0]\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-5), loss=loss_function(loss_name), metrics=["accuracy"]\n    )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\nclass HistoryBackup(keras.callbacks.Callback):\n    def __init__(self, path):\n        super().__init__()\n        self.path = path\n        self.values = {}\n\n    def on_epoch_end(self, epoch, logs=None):\n        for key, value in (logs or {}).items():\n            self.values.setdefault(key, []).append(float(value))\n        write_json(self.path, self.values)\n\n\ndef fit_stage(model, train, val, directory, name, epochs):\n    best = directory / (name + "_best.keras")\n    callbacks = [\n        keras.callbacks.ModelCheckpoint(\n            str(best), monitor="val_accuracy", mode="max", save_best_only=True\n        ),\n        keras.callbacks.CSVLogger(str(directory / (name + "_log.csv"))),\n        HistoryBackup(directory / (name + "_history.json")),\n        keras.callbacks.TerminateOnNaN(),\n    ]\n    history = model.fit(train, validation_data=val, epochs=epochs, callbacks=callbacks, verbose=2)\n    values = {key: [float(v) for v in seq] for key, seq in history.history.items()}\n    if len(values.get("val_accuracy", [])) != epochs or not all(\n        np.isfinite(seq).all() for seq in values.values()\n    ):\n        raise RuntimeError("Incomplete or non-finite training; attempt retained")\n    model.save(directory / (name + "_last.keras"))\n    return values, best\n\n\ndef checkpoint_choice(baseline_score, new_score):\n    """Keep the earlier/simpler checkpoint on ties."""\n    return new_score > baseline_score\n\n\ndef cached_record(root, trial_id, signature):\n    trial_dir = Path(root) / trial_id\n    marker = trial_dir / "completed.json"\n    if not marker.exists():\n        return None\n    record = read_json(marker)\n    if record["signature"] != signature:\n        raise ValueError("Resume settings differ; use a new suite directory")\n    for relative, digest in record["artifact_hashes"].items():\n        target = (trial_dir / relative).resolve()\n        if not target.is_relative_to(trial_dir.resolve()) or file_hash(target) != digest:\n            raise ValueError("Completed artifact changed or corrupted: " + relative)\n    return record\n\n\ndef evaluate_to_files(model, dataset, paths, directory, prefix):\n    truth, probabilities = predict_dataset(model, dataset)\n    metrics = classification_metrics(truth, probabilities, probabilities.shape[1])\n    write_json(directory / (prefix + "_metrics.json"), metrics)\n    save_predictions(directory / (prefix + "_predictions.csv"), paths, truth, probabilities)\n    return metrics\n\n\ndef run_trial(spec, dataset_factory, root, signature, seed=42, epochs1=15, epochs2=10):\n    cached = cached_record(root, spec["id"], signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / spec["id"] / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    write_json(directory / "spec.json", spec)\n    train, _ = dataset_factory("train", spec["size"], True)\n    val, paths = dataset_factory("val", spec["size"], False)\n    started = time.monotonic()\n    model = build_model(spec)\n    h1, best1 = fit_stage(model, train, val, directory, "stage1", epochs1)\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = []\n    h2 = {key: [] for key in h1}\n    best2 = best1\n    if epochs2:\n        trainable = configure_partial(model, spec["loss"])\n        h2, best2 = fit_stage(model, train, val, directory, "stage2", epochs2)\n    del model\n    selected_stage = (\n        "stage2"\n        if checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"], default=-1.0))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": spec["id"],\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]) if h2["val_accuracy"] else None,\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef finish_record(directory, record):\n    write_json(directory / "record.json", record)\n    record["artifact_hashes"] = {\n        str(path.relative_to(directory.parent)): file_hash(path)\n        for path in directory.iterdir()\n        if path.is_file()\n    }\n    write_json(directory.parent / "completed.json", record)\n\n\ndef selected_model_path(root, record):\n    return Path(root) / record["id"] / record["attempt"] / record["selected_model"]\n\n\ndef extend_b1(parent, dataset_factory, root, signature, seed=42, epochs=5):\n    trial_id = "b1_256_ls005_extend5"\n    cached = cached_record(root, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(seed)\n    directory = Path(root) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    parent_dir = Path(root) / parent["id"] / parent["attempt"]\n    # Continue from epoch 10\'s LAST checkpoint, including optimizer state.\n    source = parent_dir / "stage2_last.keras"\n    model = keras.models.load_model(source)\n    if model.optimizer is None:\n        raise ValueError("Extension requires saved optimizer")\n    train, _ = dataset_factory("train", 256, True)\n    val, paths = dataset_factory("val", 256, False)\n    started = time.monotonic()\n    history, best = fit_stage(model, train, val, directory, "extension", epochs)\n    del model\n    parent_best = selected_model_path(root, parent)\n    parent_score = max(parent["stage1_best_val"], parent["stage2_best_val"])\n    keep_extension = checkpoint_choice(parent_score, max(history["val_accuracy"]))\n    selected = directory / "selected.keras"\n    import shutil\n\n    shutil.copyfile(best if keep_extension else parent_best, selected)\n    model = keras.models.load_model(selected, compile=False)\n    metrics = evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": {**parent["spec"], "id": trial_id},\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": "extension" if keep_extension else "parent_" + parent["selected_stage"],\n        "validation": metrics,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "training_seconds": time.monotonic() - started,\n        "history": {key: parent["history"][key] + history[key] for key in history},\n        "stage_boundary": parent["stage_boundary"],\n        "extension_boundary": len(parent["history"]["accuracy"]),\n        "parent_last_sha256": file_hash(source),\n        "optimizer_restored": True,\n        "extension_best_val": max(history["val_accuracy"]),\n        "stage1_best_val": parent["stage1_best_val"],\n        "stage2_best_val": parent["stage2_best_val"],\n    }\n    finish_record(directory, record)\n    return record\n\n\ndef select_winner(records):\n    # Stable order preserves earlier experiments on exact ties.\n    return max(records, key=lambda record: record["validation"]["accuracy"])\n', 'common_workflow': '"""Standalone Colab orchestration; shared contracts, audit gates and artifacts."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\n\nEXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef signature(value):\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()\n    ).hexdigest()\n\n\ndef extract_zip(source, destination):\n    """Validate every member before creating any output; no overwrite."""\n    destination = Path(destination).resolve()\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(source) as archive:\n        seen = set()\n        for item in archive.infolist():\n            name = item.orig_filename\n            path = PurePosixPath(name)\n            if (\n                path.is_absolute()\n                or ".." in path.parts\n                or "\\\\" in name\n                or ":" in name\n                or stat.S_ISLNK(item.external_attr >> 16)\n            ):\n                raise ValueError("Unsafe ZIP member: " + name)\n            key = name.rstrip("/").casefold()\n            if key in seen:\n                raise ValueError("Duplicate ZIP destination: " + name)\n            seen.add(key)\n        destination.mkdir(parents=True)\n        archive.extractall(destination)\n\n\ndef roots_for(extracted):\n    roots = {}\n    for kind in ("original", "augmented"):\n        matches = [\n            p\n            for p in Path(extracted).rglob("*")\n            if p.is_dir()\n            and p.name.lower() == kind\n            and all((p / s).is_dir() for s in ("train", "val", "test"))\n        ]\n        if len(matches) != 1:\n            raise ValueError(f"{kind}/train,val,test 구조를 하나로 확인하세요: {matches}")\n        roots[kind] = matches[0]\n    return roots\n\n\ndef verify_audit(directory, domain, classes, digest):\n    directory = Path(directory)\n    manifest = engine.read_json(directory / "audit_manifest.json")\n    for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv"):\n        if engine.file_hash(directory / name) != manifest[name]:\n            raise ValueError("검증 보고서가 변경됐습니다: " + name)\n    audit = engine.read_json(directory / "audit_summary.json")\n    if (\n        audit["domain"] != domain\n        or audit["class_names"] != classes\n        or audit["data_sha256"] != digest\n        or audit.get("protocol") != "common_audit_v1"\n        or audit["status"] != "mechanical_checks_passed_with_limitations"\n    ):\n        raise ValueError("대상/클래스/데이터가 다르거나 검증 문제가 있습니다. 공통 ①을 확인하세요.")\n    return audit\n\n\ndef prepare(config, profiles, sources, commit, local_parent="/content"):\n    c = dict(config)\n    c.setdefault("expected_data_sha256", "")\n    c.setdefault("seeds", [c["seed"]])\n    if c["domain"] not in profiles or c["mode"] not in (\n        "audit",\n        "comparison",\n        "suite",\n        "baseline3",\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "final_candidate",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_pmg_final",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ):\n        raise ValueError("DOMAIN/MODE 설정을 확인하세요.")\n    if c["train_variant"] not in ("original", "augmented"):\n        raise ValueError("TRAIN_VARIANT는 original 또는 augmented입니다.")\n    for key in ("batch_size", "epochs1", "epochs2", "extension_epochs"):\n        if not isinstance(c[key], int) or c[key] <= 0:\n            raise ValueError(key + "는 양의 정수여야 합니다.")\n    if (\n        not isinstance(c["seeds"], list)\n        or not c["seeds"]\n        or any(not isinstance(value, int) or value < 0 for value in c["seeds"])\n        or len(set(c["seeds"])) != len(c["seeds"])\n    ):\n        raise ValueError("SEEDS는 서로 다른 0 이상의 정수 목록이어야 합니다.")\n    if c["mode"] == "baseline3" and len(c["seeds"]) != 3:\n        raise ValueError("baseline3는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "paper_suite" and len(c["seeds"]) != 3:\n        raise ValueError("paper_suite는 정확히 3개의 seed가 필요합니다.")\n    if c["mode"] == "supcon_compare" and c["seeds"] != [42]:\n        raise ValueError("supcon_compare의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] == "supcon_repeat" and c["seeds"] != [43, 44]:\n        raise ValueError("supcon_repeat의 확인 seed는 [43, 44]여야 합니다.")\n    if c["mode"] == "paper_screen" and c["seeds"] != [42]:\n        raise ValueError("paper_screen의 선별 seed는 [42]여야 합니다.")\n    if c["mode"] in (\n        "sam_screen",\n        "final_candidate",\n        "web_skin_pmg_final",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ) and c["seeds"] != [42]:\n        raise ValueError(c["mode"] + "의 seed는 [42]여야 합니다.")\n    project = Path(c["project_root"])\n    if not project.is_dir():\n        raise FileNotFoundError("PROJECT_ROOT 폴더를 확인하세요: " + str(project))\n    if c["mode"] != "audit" and not (\n        c["audit_dir"] or c["expected_data_sha256"]\n    ):\n        raise ValueError("AUDIT_DIR 또는 확인된 EXPECTED_DATA_SHA256을 입력하세요.")\n    if c["data_zip"]:\n        candidates = [Path(c["data_zip"])]\n    else:\n        candidates = sorted(\n            p\n            for p in (project / "datasets").rglob(c["domain"] + "*")\n            if p.is_file() and zipfile.is_zipfile(p)\n        )\n    if len(candidates) != 1 or not zipfile.is_zipfile(candidates[0]):\n        raise ValueError(f"DATA_ZIP으로 ZIP 하나를 지정하세요: {candidates}")\n    source = candidates[0]\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]\n    local = Path(local_parent) / ("mediflow_" + run_id)\n    local.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(source) as archive:\n        required = (\n            source.stat().st_size + sum(m.file_size for m in archive.infolist()) + 2 * 1024**3\n        )\n    if required > shutil.disk_usage(local).free:\n        raise RuntimeError("Colab 압축 해제 공간이 부족합니다.")\n    copied = local / "input.zip"\n    shutil.copyfile(source, copied)\n    digest = engine.file_hash(copied)\n    if digest != engine.file_hash(source):\n        raise OSError("Drive ZIP 복사 내용 불일치")\n    classes = profiles[c["domain"]]\n    audit = None\n    if c["mode"] != "audit":\n        if c["audit_dir"]:\n            audit = verify_audit(c["audit_dir"], c["domain"], classes, digest)\n        else:\n            expected = c["expected_data_sha256"].strip().lower()\n            if len(expected) != 64 or any(ch not in "0123456789abcdef" for ch in expected):\n                raise ValueError("EXPECTED_DATA_SHA256은 64자리 SHA-256이어야 합니다.")\n            if digest != expected:\n                raise ValueError("DATA_ZIP이 확인된 SHA-256과 다릅니다.")\n            audit = {\n                "domain": c["domain"],\n                "class_names": classes,\n                "data_sha256": digest,\n                "protocol": "expected_sha256_v1",\n                "status": "independent_audit_skipped",\n                "limitations": [\n                    "Independent common audit was skipped by the project owner",\n                    "Person, lesion and capture-session leakage remains unverified",\n                    "Perceptual near-duplicate and clinical label checks were not performed",\n                ],\n            }\n    extract_zip(copied, local / "dataset")\n    settings = {\n        k: c[k]\n        for k in (\n            "domain",\n            "mode",\n            "seed",\n            "seeds",\n            "batch_size",\n            "epochs1",\n            "epochs2",\n            "extension_epochs",\n            "train_variant",\n        )\n    }\n    settings.update(\n        classes=classes,\n        data_sha256=digest,\n        source_hashes={k: signature(v) for k, v in sources.items()},\n        protocol="common_v1",\n        audit=signature(audit),\n        environment=dict(\n            tensorflow=tf.__version__,\n            keras=keras.__version__,\n            numpy=np.__version__,\n            python=platform.python_version(),\n        ),\n    )\n    if c["mode"] in (\n        "paper_suite",\n        "supcon_compare",\n        "supcon_repeat",\n        "paper_screen",\n        "sam_screen",\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n        "web_skin_pmg_b1_384",\n        "web_skin_medsiglip_linear",\n        "hair_medsiglip_linear",\n    ):\n        settings["experiments"] = c.get("experiments", [])\n    if c["mode"] in ("sam_screen", "final_candidate", "web_skin_pmg_final"):\n        settings.update(\n            parent_run_dir=c.get("parent_run_dir"),\n            parent_model_sha256=c.get(\n                "parent_model_sha256", c.get("parent_stage1_sha256")\n            ),\n        )\n    if c["mode"] == "sam_screen":\n        settings["sam_rho"] = c.get("sam_rho")\n    if c["mode"] in ("web_skin_wsdan", "web_skin_paper_suite"):\n        settings.update(\n            attention_maps=c.get("attention_maps"),\n            crop_threshold=c.get("crop_threshold"),\n            drop_threshold=c.get("drop_threshold"),\n        )\n    if c["mode"] in ("web_skin_paper_suite", "web_skin_pmg_b1_384"):\n        settings.update(\n            pmg_jigsaw_grids=c.get("pmg_jigsaw_grids"),\n        )\n    if c["mode"] == "web_skin_paper_suite":\n        settings.update(\n            mixstyle_alpha=c.get("mixstyle_alpha"),\n            mixstyle_probability=c.get("mixstyle_probability"),\n        )\n    if c["mode"] in ("web_skin_medsiglip_linear", "hair_medsiglip_linear"):\n        settings.update(\n            model_id=c.get("model_id"),\n            embedding_shard_size=c.get("embedding_shard_size"),\n            linear_batch_size=c.get("linear_batch_size"),\n            linear_learning_rate=c.get("linear_learning_rate"),\n            linear_weight_decay=c.get("linear_weight_decay"),\n        )\n    sig = signature(settings)\n    if c["resume_dir"]:\n        output = Path(c["resume_dir"])\n        if c["mode"] == "audit":\n            raise ValueError("검사는 새 실행으로 시작하세요. RESUME_DIR을 비우세요.")\n        if engine.read_json(output / "run_config.json")["signature"] != sig:\n            raise ValueError(\n                "코드/설정/환경/데이터/검증이 다른 실행입니다. 새 결과 폴더를 사용하세요."\n            )\n    else:\n        output = project / "2_results" / c["domain"] / (c["mode"] + "_" + run_id)\n        output.mkdir(parents=True, exist_ok=False)\n        engine.write_json(\n            output / "run_config.json",\n            {\n                "signature": sig,\n                "settings": settings,\n                "code_commit_at_generation": commit,\n                "code_state": "embedded sources include uncommitted changes; exact sources saved",\n                "source_zip": str(source),\n                "audit_source": c["audit_dir"] or "expected_data_sha256_only",\n                "baseline": (\n                    (\n                        "Fixed PMG B0/256 Validation candidate reused; model not retrained"\n                        if c["mode"]\n                        in ("web_skin_pmg_final", "web_skin_medsiglip_linear")\n                        else (\n                            "Fixed Hair B1/384 Validation candidate reused; model not retrained"\n                            if c["mode"] == "hair_medsiglip_linear"\n                            else "Saved B0/256/CE Validation metrics reused; baseline not retrained"\n                        )\n                    )\n                    if c["mode"]\n                    in (\n                        "web_skin_wsdan",\n                        "web_skin_paper_suite",\n                        "web_skin_pmg_b1_384",\n                        "web_skin_pmg_final",\n                        "web_skin_medsiglip_linear",\n                        "hair_medsiglip_linear",\n                    )\n                    else "ImageNet pretrained EfficientNet; historical metrics not reused"\n                ),\n                "gpu": [str(d) for d in tf.config.list_physical_devices("GPU")],\n            },\n        )\n        for name, code in sources.items():\n            (output / (name + ".py")).write_text(code, encoding="utf-8")\n        engine.write_json(output / "class_names.json", classes)\n        if c["audit_dir"]:\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "audit_summary.json", output / "audit_summary.json"\n            )\n            shutil.copyfile(\n                Path(c["audit_dir"]) / "image_inventory.csv", output / "image_inventory.csv"\n            )\n        elif audit:\n            engine.write_json(output / "audit_summary.json", audit)\n    context = dict(\n        config=c,\n        classes=classes,\n        signature=sig,\n        output=output,\n        local=local,\n        data_hash=digest,\n        audit=audit,\n        extracted=local / "dataset",\n        project=project,\n    )\n    if c["mode"] != "audit":\n        context["roots"] = roots_for(context["extracted"])\n        # Dataset ZIP is immutable and matches the audit; verify class folders again.\n        for root in context["roots"].values():\n            for split in ("train", "val", "test"):\n                found = sorted(p.name for p in (root / split).iterdir() if p.is_dir())\n                if found != sorted(classes):\n                    raise ValueError(f"클래스 불일치: {root / split}")\n    print("실행 결과:", output)\n    return context\n\n\ndef archive_results(context):\n    output = context["output"]\n    destination = output.parent / (output.name + "_results_" + uuid.uuid4().hex[:8] + ".zip")\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(output.rglob("*")):\n            if p.is_file() and p.suffix not in (".keras", ".tmp"):\n                archive.write(p, output.name + "/" + p.relative_to(output).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("보고서 ZIP 검사 실패")\n    print("로컬로 내려받을 결과 ZIP:", destination)\n    return destination\n\n\ndef audit_run(context):\n    from mediflow_datasets.common_audit import audit_dataset\n\n    summary = audit_dataset(\n        context["extracted"],\n        context["output"],\n        context["config"]["domain"],\n        context["classes"],\n        context["data_hash"],\n    )\n    engine.write_json(\n        context["output"] / "audit_manifest.json",\n        {\n            name: engine.file_hash(context["output"] / name)\n            for name in ("audit_summary.json", "image_inventory.csv", "audit_issues.csv")\n        },\n    )\n    archive_results(context)\n    print("검사 상태:", summary["status"], "\\n학습 AUDIT_DIR:", context["output"])\n    return summary\n\n\ndef factory(context, variant, seed=None):\n    shuffle_seed = context["config"]["seed"] if seed is None else seed\n\n    def load(split, size, shuffle):\n        # All trials use exactly the same ORIGINAL validation and test images.\n        root = context["roots"][variant if split == "train" else "original"]\n        ds = keras.utils.image_dataset_from_directory(\n            root / split,\n            class_names=context["classes"],\n            label_mode="categorical",\n            image_size=(size, size),\n            interpolation="bilinear",\n            batch_size=context["config"]["batch_size"],\n            shuffle=shuffle,\n            seed=shuffle_seed if shuffle else None,\n        )\n        paths = [Path(p).relative_to(context["extracted"]).as_posix() for p in ds.file_paths]\n        return ds.prefetch(tf.data.AUTOTUNE), paths\n\n    return load\n\n\ndef run(context):\n    c, output = context["config"], context["output"]\n    if c["mode"] == "comparison":\n        trials = [\n            dict(id=kind, backbone="B0", size=224, loss="ce", variant=kind)\n            for kind in ("original", "augmented")\n        ]\n    else:\n        trials = [dict(spec, variant=c["train_variant"]) for spec in engine.TRIALS]\n    records = []\n    try:\n        for spec in trials:\n            spec["class_count"] = len(context["classes"])\n            spec["class_names"] = context["classes"]\n            load = factory(context, spec["variant"])\n            record = engine.run_trial(\n                spec,\n                load,\n                output,\n                context["signature"],\n                c["seed"],\n                c["epochs1"],\n                0 if c["mode"] == "comparison" else c["epochs2"],\n            )\n            records.append(record)\n            engine.write_json(output / "progress.json", {"completed": [r["id"] for r in records]})\n        if c["mode"] == "suite":\n            parent = records[-1]\n            records.append(\n                engine.extend_b1(\n                    parent,\n                    factory(context, c["train_variant"]),\n                    output,\n                    context["signature"],\n                    c["seed"],\n                    c["extension_epochs"],\n                )\n            )\n        engine.write_json(output / "all_validation_results.json", records)\n        return records\n    except Exception as exc:\n        engine.write_json(\n            output / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [r["id"] for r in records],\n                "resume_dir": str(output),\n            },\n        )\n        print("중단. 완료된 실험을 유지합니다. RESUME_DIR:", output)\n        raise\n\n\ndef confusion(ax, metrics, title):\n    cm = np.asarray(metrics["confusion_matrix"])\n    ax.imshow(cm, cmap="Blues")\n    codes = [f"C{i}" for i in range(len(cm))]\n    ax.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(cm)),\n        yticks=range(len(cm)),\n        xticklabels=codes,\n        yticklabels=codes,\n    )\n    for i in range(len(cm)):\n        for j in range(len(cm)):\n            ax.text(\n                j,\n                i,\n                str(cm[i, j]),\n                ha="center",\n                va="center",\n                fontsize=8,\n                color="white" if cm[i, j] > cm.max() / 2 else "black",\n            )\n\n\ndef errors(context, csv_path, destination):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    from PIL import Image\n\n    frame = pd.read_csv(csv_path)\n    wrong = frame[frame.true_index != frame.pred_index].head(8)\n    fig, axes = plt.subplots(2, 4, figsize=(14, 7))\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, (_, row) in zip(axes.flat, wrong.iterrows(), strict=False):\n        path = (context["extracted"] / row["path"]).resolve()\n        if not path.is_relative_to(context["extracted"].resolve()):\n            raise ValueError("Prediction path escapes dataset")\n        with Image.open(path) as image:\n            ax.imshow(image.convert("RGB"))\n        ax.set_title(f"True C{row.true_index} / Pred C{row.pred_index}")\n    if wrong.empty:\n        fig.suptitle("No misclassifications")\n    fig.tight_layout()\n    fig.savefig(destination, dpi=160)\n    plt.close(fig)\n\n\ndef overview(context, records):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    output = context["output"]\n    fig, axes = plt.subplots(\n        (len(records) + 1) // 2, 4, figsize=(24, 4.5 * ((len(records) + 1) // 2)), squeeze=False\n    )\n    for index, record in enumerate(records):\n        row, col = divmod(index, 2)\n        for offset, metric in enumerate(("accuracy", "loss")):\n            ax = axes[row, col * 2 + offset]\n            h = record["history"]\n            x = np.arange(1, len(h[metric]) + 1)\n            ax.plot(x, h[metric], label="Train")\n            ax.plot(x, h["val_" + metric], label="Validation")\n            if record["stage_boundary"] < len(x):\n                ax.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n            if "extension_boundary" in record:\n                ax.axvline(record["extension_boundary"] + 0.5, ls=":", color="green")\n            ax.set(title=record["id"] + " / " + metric, xlabel="Epoch", ylabel=metric)\n            if metric == "accuracy":\n                ax.set_ylim(0, 1)\n            ax.grid(alpha=0.25)\n            ax.legend()\n        directory = output / record["id"] / record["attempt"]\n        errors(\n            context, directory / "validation_predictions.csv", directory / "validation_errors.png"\n        )\n    fig.suptitle(context["config"]["domain"] + " / Loss definitions differ across CE, LS, Focal")\n    fig.tight_layout()\n    fig.savefig(output / "all_training_curves.png", dpi=180)\n    fig.savefig(output / "all_training_curves.pdf")\n    plt.show()\n    plt.close(fig)\n    table = pd.DataFrame(\n        [\n            dict(\n                experiment=r["id"],\n                selected_stage=r["selected_stage"],\n                validation_accuracy=r["validation"]["accuracy"],\n                validation_macro_f1=r["validation"]["macro_f1"],\n                parameters=r["parameters"],\n                seconds_this_trial=r["training_seconds"],\n                epochs=len(r["history"]["accuracy"]),\n                model_bytes=r["model_bytes"],\n            )\n            for r in records\n        ]\n    )\n    table.to_csv(output / "experiment_comparison.csv", index=False, encoding="utf-8-sig")\n    print(table.to_string(index=False))\n    fig, axes = plt.subplots(2, 1, figsize=(14, 11))\n    x = np.arange(len(records))\n    axes[0].bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axes[0].bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    axes[0].set(xticks=x, xticklabels=table.experiment, ylim=(0, 1))\n    axes[0].tick_params(axis="x", labelrotation=15)\n    axes[0].legend()\n    matrix = np.array([r["validation"]["class_f1"] for r in records])\n    axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")\n    axes[1].set(\n        xticks=range(len(context["classes"])),\n        xticklabels=[f"C{i}" for i in range(len(context["classes"]))],\n        yticks=x,\n        yticklabels=table.experiment,\n        title="Validation class F1",\n    )\n    for i in range(len(records)):\n        for j in range(len(context["classes"])):\n            axes[1].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=7)\n    fig.tight_layout()\n    fig.savefig(output / "validation_performance_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    fig, axes = plt.subplots(\n        (len(records) + 2) // 3, 3, figsize=(18, 6 * ((len(records) + 2) // 3)), squeeze=False\n    )\n    for ax in axes.flat:\n        ax.axis("off")\n    for ax, r in zip(axes.flat, records, strict=False):\n        ax.axis("on")\n        confusion(ax, r["validation"], r["id"])\n    fig.tight_layout()\n    fig.savefig(output / "all_validation_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef finish(context, records):\n    import matplotlib.pyplot as plt\n\n    output = context["output"]\n    overview(context, records)\n    winner = engine.select_winner(records)\n    model_path = engine.selected_model_path(output, winner)\n    selection = dict(\n        winner=winner["id"],\n        model_sha256=engine.file_hash(model_path),\n        signature=context["signature"],\n        validation=winner["validation"],\n    )\n    selected_file = output / "selection_before_test.json"\n    if selected_file.exists() and engine.read_json(selected_file) != selection:\n        raise ValueError("이미 고정한 선택 모델이 다릅니다.")\n    if not selected_file.exists():\n        engine.write_json(selected_file, selection)\n    marker = output / "test_completed.json"\n    if marker.exists():\n        tested = engine.read_json(marker)\n        if tested["selection"] != selection:\n            raise ValueError("기존 Test 모델과 다릅니다.")\n        for name, digest in tested["hashes"].items():\n            if engine.file_hash(output / name) != digest:\n                raise ValueError("Test 파일이 변경됐습니다.")\n        metrics = tested["metrics"]\n    else:\n        keras.backend.clear_session()\n        model = keras.models.load_model(model_path, compile=False)\n        ds, paths = factory(context, winner["spec"]["variant"])(\n            "test", winner["spec"]["size"], False\n        )\n        metrics = engine.evaluate_to_files(model, ds, paths, output, "final_test")\n        engine.write_json(\n            marker,\n            dict(\n                selection=selection,\n                metrics=metrics,\n                hashes={\n                    name: engine.file_hash(output / name)\n                    for name in ("final_test_metrics.json", "final_test_predictions.csv")\n                },\n            ),\n        )\n        del model\n    fig, ax = plt.subplots(figsize=(8, 8))\n    confusion(ax, metrics, winner["id"] + " / Final Test")\n    fig.tight_layout()\n    fig.savefig(output / "final_test_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    errors(context, output / "final_test_predictions.csv", output / "final_test_errors.png")\n    card = dict(\n        domain=context["config"]["domain"],\n        class_names=context["classes"],\n        normal_included="정상" in context["classes"],\n        validation=winner["validation"],\n        test=metrics,\n        selected_model=winner["id"],\n        model_sha256=selection["model_sha256"],\n        input_size=winner["spec"]["size"],\n        data_sha256=context["data_hash"],\n        status="public_data_candidate_not_device_validated",\n        limitations=context["audit"]["limitations"]\n        + [\n            "Single seed; small differences are not established as robust gains",\n            "Out-of-scope rejection absent; scores are not calibrated correctness",\n        ],\n    )\n    engine.write_json(output / "model_card.json", card)\n    package_parent = (\n        context["project"] / "2_results" / context["config"]["domain"] / "selected_models"\n    )\n    package = package_parent / (output.name + "_" + uuid.uuid4().hex[:8])\n    package.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(model_path, package / "model.keras")\n    if engine.file_hash(package / "model.keras") != selection["model_sha256"]:\n        raise OSError("모델 복사 불일치")\n    for name in (\n        "class_names.json",\n        "model_card.json",\n        "run_config.json",\n        "selection_before_test.json",\n        "audit_summary.json",\n        "final_test_metrics.json",\n        "common_engine.py",\n        "common_audit.py",\n        "common_workflow.py",\n    ):\n        shutil.copyfile(output / name, package / name)\n    engine.write_json(\n        package / "preprocessing.json",\n        dict(\n            input_shape=[winner["spec"]["size"], winner["spec"]["size"], 3],\n            color="RGB",\n            dtype="float32",\n            pixel_range=[0, 255],\n            external_normalization=False,\n            internal_rescaling="1/255",\n            resize="TensorFlow bilinear; no crop/pad; antialias=False",\n            exif_transpose=False,\n            output="softmax scores in class_names.json order",\n        ),\n    )\n    engine.write_json(\n        package / "manifest.json",\n        {p.name: engine.file_hash(p) for p in package.iterdir() if p.is_file()},\n    )\n    destination = Path(shutil.make_archive(str(package), "zip", package.parent, package.name))\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip():\n            raise OSError("모델 ZIP 손상")\n        manifest = engine.read_json(package / "manifest.json")\n        for name, digest in manifest.items():\n            if hashlib.sha256(archive.read(package.name + "/" + name)).hexdigest() != digest:\n                raise OSError("ZIP 내용 불일치: " + name)\n    destination.with_suffix(".zip.sha256").write_text(\n        engine.file_hash(destination), encoding="ascii"\n    )\n    archive_results(context)\n    print("선정 모델:", winner["id"], "\\nTest:", metrics, "\\n후보 ZIP:", destination)\n    return card\n', 'web_skin_wsdan': '"""Validation-only WS-DAN-inspired screening for Web Skin.\n\nThe implementation adapts Hu et al. (2019) to EfficientNet-B0 with bilinear\nattention pooling and attention-guided crop/drop augmentation.  It uses image\nlabels only and deliberately does not access Test during screening.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\nfrom mediflow_datasets import common_workflow as flow\n\nPROTOCOL = "web_skin_wsdan_screen_v1"\nTRIAL_ID = "wsdan_b0_256_ce_seed_42"\nPAPER_URL = "https://arxiv.org/abs/1901.09891"\nEXPECTED_CLASSES = ["건선", "아토피", "여드름", "정상", "주사"]\nBASELINE = {\n    "id": "b0_256_ce",\n    "validation_accuracy": 0.796,\n    "validation_macro_f1": 0.7915394647566528,\n    "validation_count": 500,\n    "class_f1": [\n        0.7606837606837609,\n        0.6847826086956522,\n        0.6818181818181819,\n        0.9345794392523363,\n        0.8958333333333334,\n    ],\n    "confusion_matrix": [\n        [89, 3, 3, 3, 2],\n        [18, 63, 8, 8, 3],\n        [22, 14, 60, 3, 1],\n        [0, 0, 0, 100, 0],\n        [5, 4, 5, 0, 86],\n    ],\n    "data_sha256": "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d",\n    "source": (\n        "results/web_skin/experiments/suite_20260908_014452_72768a42/"\n        "b0_256_ce/attempt_8977c45d04bc/validation_metrics.json"\n    ),\n}\n\n\ndef validate(context):\n    config = context["config"]\n    if config["domain"] != "web_skin" or config["mode"] not in (\n        "web_skin_wsdan",\n        "web_skin_paper_suite",\n    ):\n        raise ValueError("이 노트북은 Web Skin WS-DAN 선별 전용입니다.")\n    if config["train_variant"] != "augmented":\n        raise ValueError("기존 기준선과 같은 augmented Train을 사용해야 합니다.")\n    if config["seed"] != 42 or config["seeds"] != [42]:\n        raise ValueError("WS-DAN 선별 seed는 42로 고정합니다.")\n    experiments = config.get("experiments", [])\n    if config["mode"] == "web_skin_wsdan" and experiments != [TRIAL_ID]:\n        raise ValueError("이번 실행은 WS-DAN 한 조건만 허용합니다.")\n    if config["mode"] == "web_skin_paper_suite" and TRIAL_ID not in experiments:\n        raise ValueError("통합 실험에 WS-DAN 조건이 없습니다.")\n    if context["classes"] != EXPECTED_CLASSES:\n        raise ValueError("Web Skin 클래스 순서가 기존 모델 계약과 다릅니다.")\n    if context.get("data_hash") != BASELINE["data_sha256"]:\n        raise ValueError("저장된 B0·256·CE 기준선과 데이터 SHA-256이 다릅니다.")\n    maps = config.get("attention_maps")\n    if not isinstance(maps, int) or maps <= 0:\n        raise ValueError("ATTENTION_MAPS는 양의 정수여야 합니다.")\n    for key in ("crop_threshold", "drop_threshold"):\n        value = config.get(key)\n        if not isinstance(value, (int, float)) or not 0 < value < 1:\n            raise ValueError(key + "는 0과 1 사이여야 합니다.")\n\n\n@keras.saving.register_keras_serializable(package="MediFlow")\nclass BilinearAttentionPooling(keras.layers.Layer):\n    """Pool one feature vector per learned attention map."""\n\n    def call(self, inputs):\n        features, attention = inputs\n        pooled = keras.ops.einsum("bhwc,bhwm->bmc", features, attention)\n        normalizer = keras.ops.sum(attention, axis=(1, 2))\n        pooled = pooled / (keras.ops.expand_dims(normalizer, -1) + 1e-6)\n        pooled = keras.ops.reshape(pooled, (keras.ops.shape(pooled)[0], -1))\n        pooled = keras.ops.sign(pooled) * keras.ops.sqrt(keras.ops.abs(pooled) + 1e-8)\n        norm = keras.ops.sqrt(\n            keras.ops.sum(keras.ops.square(pooled), axis=-1, keepdims=True) + 1e-8\n        )\n        return pooled / norm\n\n    def compute_output_shape(self, input_shape):\n        feature_shape, attention_shape = input_shape\n        return (feature_shape[0], feature_shape[-1] * attention_shape[-1])\n\n\ndef build_model(class_count, size=256, attention_maps=8, weights="imagenet"):\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3), name="image")\n    features = backbone(inputs, training=False)\n    attention = keras.layers.Conv2D(\n        attention_maps, 1, activation="sigmoid", name="attention_maps"\n    )(features)\n    pooled = BilinearAttentionPooling(name="bilinear_attention_pooling")(\n        [features, attention]\n    )\n    pooled = keras.layers.Dropout(0.3, name="attention_dropout")(pooled)\n    outputs = keras.layers.Dense(class_count, activation="softmax", name="predictions")(\n        pooled\n    )\n    model = keras.Model(inputs, outputs, name="web_skin_wsdan_b0")\n    attention_probe = keras.Model(inputs, [outputs, attention], name="attention_probe")\n    return model, attention_probe\n\n\ndef configure_partial(model):\n    backbones = [\n        layer\n        for layer in model.layers\n        if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()\n    ]\n    if len(backbones) != 1:\n        raise ValueError("EfficientNet Backbone 하나가 필요합니다.")\n    backbone = backbones[0]\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\ndef _one_attention_augmentation(item, size, crop_threshold, drop_threshold):\n    image, maps = item\n    score = tf.reduce_mean(maps, axis=-1, keepdims=True)\n    score = tf.image.resize(score, (size, size), method="bilinear")\n    score = score / (tf.reduce_max(score) + 1e-6)\n    crop_mask = score[..., 0] >= crop_threshold\n    coordinates = tf.cast(tf.where(crop_mask), tf.int32)\n\n    def crop_region():\n        top_left = tf.reduce_min(coordinates, axis=0)\n        bottom_right = tf.reduce_max(coordinates, axis=0) + 1\n        height = tf.maximum(bottom_right[0] - top_left[0], 1)\n        width = tf.maximum(bottom_right[1] - top_left[1], 1)\n        crop = tf.image.crop_to_bounding_box(\n            image, top_left[0], top_left[1], height, width\n        )\n        return tf.image.resize(crop, (size, size), method="bilinear")\n\n    crop = tf.cond(tf.shape(coordinates)[0] > 0, crop_region, lambda: image)\n    keep = tf.cast(score < drop_threshold, image.dtype)\n    dropped = image * keep\n    choose_crop = tf.random.uniform(()) < 0.5\n    return tf.cond(choose_crop, lambda: crop, lambda: dropped)\n\n\ndef attention_augment(images, maps, size, crop_threshold, drop_threshold):\n    maps = tf.stop_gradient(maps)\n    return tf.map_fn(\n        lambda item: _one_attention_augmentation(\n            item, size, crop_threshold, drop_threshold\n        ),\n        (images, maps),\n        fn_output_signature=tf.TensorSpec((size, size, 3), images.dtype),\n    )\n\n\ndef _one_attention_crop(item, size, crop_threshold):\n    image, maps = item\n    score = tf.reduce_mean(maps, axis=-1, keepdims=True)\n    score = tf.image.resize(score, (size, size), method="bilinear")\n    score = score / (tf.reduce_max(score) + 1e-6)\n    coordinates = tf.cast(tf.where(score[..., 0] >= crop_threshold), tf.int32)\n\n    def crop_region():\n        top_left = tf.reduce_min(coordinates, axis=0)\n        bottom_right = tf.reduce_max(coordinates, axis=0) + 1\n        crop = tf.image.crop_to_bounding_box(\n            image,\n            top_left[0],\n            top_left[1],\n            tf.maximum(bottom_right[0] - top_left[0], 1),\n            tf.maximum(bottom_right[1] - top_left[1], 1),\n        )\n        return tf.image.resize(crop, (size, size), method="bilinear")\n\n    return tf.cond(tf.shape(coordinates)[0] > 0, crop_region, lambda: image)\n\n\ndef attention_crop(images, maps, size, crop_threshold):\n    return tf.map_fn(\n        lambda item: _one_attention_crop(item, size, crop_threshold),\n        (images, maps),\n        fn_output_signature=tf.TensorSpec((size, size, 3), images.dtype),\n    )\n\n\nclass AttentionEnsemblePredictor:\n    def __init__(self, model, size, crop_threshold):\n        self.model = model\n        self.size = size\n        self.crop_threshold = crop_threshold\n        self.probe = keras.Model(\n            model.input,\n            [model.output, model.get_layer("attention_maps").output],\n        )\n\n    def __call__(self, images, training=False):\n        raw, maps = self.probe(images, training=False)\n        cropped_images = attention_crop(\n            images, maps, self.size, self.crop_threshold\n        )\n        cropped = self.model(cropped_images, training=False)\n        return (raw + cropped) / 2.0\n\n\ndef _fit_stage(\n    model,\n    attention_probe,\n    train,\n    val,\n    directory,\n    name,\n    epochs,\n    learning_rate,\n    size,\n    crop_threshold,\n    drop_threshold,\n):\n    optimizer = keras.optimizers.Adam(learning_rate)\n    loss_function = keras.losses.CategoricalCrossentropy()\n    history = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}\n    best_score = -1.0\n    best = directory / f"{name}_best.keras"\n    log_path = directory / f"{name}_log.csv"\n\n    @tf.function\n    def train_step(images, labels):\n        with tf.GradientTape() as tape:\n            raw_predictions, maps = attention_probe(images, training=True)\n            augmented = attention_augment(\n                images, maps, size, crop_threshold, drop_threshold\n            )\n            augmented_predictions = model(augmented, training=True)\n            raw_loss = loss_function(labels, raw_predictions)\n            augmented_loss = loss_function(labels, augmented_predictions)\n            loss = 0.5 * (raw_loss + augmented_loss)\n        gradients = tape.gradient(loss, model.trainable_variables)\n        pairs = [\n            (gradient, variable)\n            for gradient, variable in zip(\n                gradients, model.trainable_variables, strict=True\n            )\n            if gradient is not None\n        ]\n        optimizer.apply_gradients(pairs)\n        return loss, raw_predictions\n\n    @tf.function\n    def validation_step(images, labels):\n        raw_predictions, maps = attention_probe(images, training=False)\n        cropped_images = attention_crop(images, maps, size, crop_threshold)\n        cropped_predictions = model(cropped_images, training=False)\n        predictions = (raw_predictions + cropped_predictions) / 2.0\n        return loss_function(labels, predictions), predictions\n\n    with log_path.open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(\n            stream, fieldnames=["epoch", "accuracy", "loss", "val_accuracy", "val_loss"]\n        )\n        writer.writeheader()\n        for epoch in range(epochs):\n            train_loss = keras.metrics.Mean()\n            train_accuracy = keras.metrics.CategoricalAccuracy()\n            val_loss = keras.metrics.Mean()\n            val_accuracy = keras.metrics.CategoricalAccuracy()\n            for images, labels in train:\n                loss, predictions = train_step(images, labels)\n                train_loss.update_state(loss)\n                train_accuracy.update_state(labels, predictions)\n            for images, labels in val:\n                loss, predictions = validation_step(images, labels)\n                val_loss.update_state(loss)\n                val_accuracy.update_state(labels, predictions)\n            values = {\n                "accuracy": float(train_accuracy.result()),\n                "loss": float(train_loss.result()),\n                "val_accuracy": float(val_accuracy.result()),\n                "val_loss": float(val_loss.result()),\n            }\n            for key, value in values.items():\n                history[key].append(value)\n            writer.writerow({"epoch": epoch + 1, **values})\n            stream.flush()\n            engine.write_json(directory / f"{name}_history.json", history)\n            print(\n                f"{name} Epoch {epoch + 1}/{epochs} - loss: {values[\'loss\']:.4f} "\n                f"- accuracy: {values[\'accuracy\']:.4f} - val_loss: "\n                f"{values[\'val_loss\']:.4f} - val_accuracy: {values[\'val_accuracy\']:.4f}"\n            )\n            if values["val_accuracy"] > best_score:\n                best_score = values["val_accuracy"]\n                model.save(best)\n    if not best.is_file() or len(history["val_accuracy"]) != epochs:\n        raise RuntimeError("WS-DAN 학습이 완전하게 끝나지 않았습니다.")\n    model.save(directory / f"{name}_last.keras")\n    return history, best\n\n\ndef run(context):\n    validate(context)\n    config = context["config"]\n    spec = {\n        "id": TRIAL_ID,\n        "method": "wsdan_inspired_attention_crop_drop",\n        "paper": PAPER_URL,\n        "backbone": "B0",\n        "size": 256,\n        "loss": "ce",\n        "variant": "augmented",\n        "epochs1": config["epochs1"],\n        "epochs2": config["epochs2"],\n        "attention_maps": config["attention_maps"],\n        "crop_threshold": float(config["crop_threshold"]),\n        "drop_threshold": float(config["drop_threshold"]),\n        "seed": 42,\n        "class_count": len(context["classes"]),\n        "class_names": context["classes"],\n        "protocol": PROTOCOL,\n        "test_evaluated": False,\n    }\n    signature = hashlib.sha256(\n        (context["signature"] + json.dumps(spec, sort_keys=True)).encode()\n    ).hexdigest()\n    cached = engine.cached_record(context["output"], TRIAL_ID, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(42)\n    directory = Path(context["output"]) / TRIAL_ID / ("attempt_" + uuid.uuid4().hex[:12])\n    directory.mkdir(parents=True, exist_ok=False)\n    engine.write_json(directory / "spec.json", spec)\n    engine.write_json(Path(context["output"]) / "baseline_reference.json", BASELINE)\n    train, _ = flow.factory(context, "augmented", seed=42)("train", 256, True)\n    val, paths = flow.factory(context, "augmented", seed=42)("val", 256, False)\n    started = time.monotonic()\n    model, probe = build_model(\n        len(context["classes"]), 256, config["attention_maps"]\n    )\n    h1, best1 = _fit_stage(\n        model,\n        probe,\n        train,\n        val,\n        directory,\n        "stage1",\n        config["epochs1"],\n        1e-4,\n        256,\n        config["crop_threshold"],\n        config["drop_threshold"],\n    )\n    del model, probe\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    probe = keras.Model(\n        model.input,\n        [model.output, model.get_layer("attention_maps").output],\n        name="attention_probe",\n    )\n    trainable = configure_partial(model)\n    h2, best2 = _fit_stage(\n        model,\n        probe,\n        train,\n        val,\n        directory,\n        "stage2",\n        config["epochs2"],\n        1e-5,\n        256,\n        config["crop_threshold"],\n        config["drop_threshold"],\n    )\n    selected_stage = (\n        "stage2"\n        if engine.checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"]))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    del model, probe\n    keras.backend.clear_session()\n    model = keras.models.load_model(selected, compile=False)\n    predictor = AttentionEnsemblePredictor(\n        model, 256, float(config["crop_threshold"])\n    )\n    metrics = engine.evaluate_to_files(\n        predictor, val, paths, directory, "validation"\n    )\n    record = {\n        "id": TRIAL_ID,\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]),\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n        "test_evaluated": False,\n        "inference": "mean of original and deterministic attention-crop predictions",\n    }\n    engine.finish_record(directory, record)\n    return record\n\n\ndef _attention_figure(model, dataset, directory, classes):\n    import matplotlib.pyplot as plt\n\n    probe = keras.Model(\n        model.input,\n        [model.output, model.get_layer("attention_maps").output],\n    )\n    images, labels = next(iter(dataset))\n    predictions, maps = probe(images[:12], training=False)\n    heatmaps = tf.reduce_mean(maps, axis=-1, keepdims=True)\n    heatmaps = tf.image.resize(heatmaps, (256, 256)).numpy()\n    fig, axes = plt.subplots(3, 4, figsize=(14, 10))\n    for index, axis in enumerate(axes.flat):\n        image = np.clip(images[index].numpy() / 255.0, 0, 1)\n        heat = heatmaps[index, ..., 0]\n        heat = heat / (heat.max() + 1e-6)\n        axis.imshow(image)\n        axis.imshow(heat, cmap="jet", alpha=0.38, vmin=0, vmax=1)\n        true_name = classes[int(tf.argmax(labels[index]))]\n        pred_name = classes[int(tf.argmax(predictions[index]))]\n        axis.set_title(f"정답: {true_name}\\n예측: {pred_name}", fontsize=9)\n        axis.axis("off")\n    fig.suptitle("Web Skin WS-DAN Attention Map", fontsize=15)\n    fig.tight_layout()\n    fig.savefig(directory / "wsdan_attention_examples.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef _validation_figures(record, output, classes):\n    import matplotlib.pyplot as plt\n\n    wsdan = record["validation"]\n    methods = ["Saved B0 256 CE", "WS-DAN B0 256 CE"]\n    accuracy = [BASELINE["validation_accuracy"], wsdan["accuracy"]]\n    macro_f1 = [BASELINE["validation_macro_f1"], wsdan["macro_f1"]]\n    positions = np.arange(len(methods))\n    width = 0.34\n    fig, axes = plt.subplots(2, 1, figsize=(13, 10))\n    axes[0].bar(positions - width / 2, accuracy, width, label="Accuracy")\n    axes[0].bar(positions + width / 2, macro_f1, width, label="Macro F1")\n    for index, value in enumerate(accuracy):\n        axes[0].text(index - width / 2, value + 0.004, f"{value:.4f}", ha="center")\n    for index, value in enumerate(macro_f1):\n        axes[0].text(index + width / 2, value + 0.004, f"{value:.4f}", ha="center")\n    axes[0].set_xticks(positions, methods)\n    axes[0].set_ylim(0, 1)\n    axes[0].set_ylabel("Validation score")\n    axes[0].set_title("Web Skin Validation Performance")\n    axes[0].legend()\n    axes[0].grid(axis="y", alpha=0.25)\n\n    class_positions = np.arange(len(classes))\n    axes[1].bar(\n        class_positions - width / 2,\n        BASELINE["class_f1"],\n        width,\n        label="Saved B0 256 CE",\n    )\n    axes[1].bar(\n        class_positions + width / 2,\n        wsdan["class_f1"],\n        width,\n        label="WS-DAN B0 256 CE",\n    )\n    axes[1].set_xticks(class_positions, [f"C{i}" for i in class_positions])\n    axes[1].set_ylim(0, 1)\n    axes[1].set_ylabel("Validation class F1")\n    axes[1].set_title("Class F1 (C0-C4 follow class_mapping.json)")\n    axes[1].legend()\n    axes[1].grid(axis="y", alpha=0.25)\n    fig.tight_layout()\n    fig.savefig(output / "wsdan_validation_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    matrix = np.asarray(wsdan["confusion_matrix"], dtype=int)\n    fig, axis = plt.subplots(figsize=(8, 7))\n    image = axis.imshow(matrix, cmap="Blues")\n    for row in range(matrix.shape[0]):\n        for column in range(matrix.shape[1]):\n            axis.text(\n                column,\n                row,\n                str(matrix[row, column]),\n                ha="center",\n                va="center",\n                color="white" if matrix[row, column] > matrix.max() / 2 else "black",\n            )\n    labels = [f"C{i}" for i in range(len(classes))]\n    axis.set_xticks(range(len(classes)), labels)\n    axis.set_yticks(range(len(classes)), labels)\n    axis.set_xlabel("Predicted class")\n    axis.set_ylabel("True class")\n    axis.set_title("WS-DAN Validation Confusion Matrix")\n    fig.colorbar(image, ax=axis)\n    fig.tight_layout()\n    fig.savefig(output / "wsdan_validation_confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n\ndef summarize(context, record):\n    import matplotlib.pyplot as plt\n\n    validate(context)\n    output = Path(context["output"])\n    attempt = output / record["id"] / record["attempt"]\n    model = keras.models.load_model(attempt / record["selected_model"], compile=False)\n    val, _ = flow.factory(context, "augmented", seed=42)("val", 256, False)\n    _attention_figure(model, val, output, context["classes"])\n    _validation_figures(record, output, context["classes"])\n    rows = [\n        {\n            "method": "saved_b0_256_ce_baseline",\n            "validation_accuracy": BASELINE["validation_accuracy"],\n            "validation_macro_f1": BASELINE["validation_macro_f1"],\n            "trained_in_this_run": False,\n        },\n        {\n            "method": "wsdan_b0_256_ce",\n            "validation_accuracy": record["validation"]["accuracy"],\n            "validation_macro_f1": record["validation"]["macro_f1"],\n            "trained_in_this_run": True,\n        },\n    ]\n    with (output / "wsdan_comparison.csv").open(\n        "w", encoding="utf-8-sig", newline=""\n    ) as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    summary = {\n        "protocol": PROTOCOL,\n        "paper": PAPER_URL,\n        "implementation": (\n            "EfficientNet-B0 adaptation with bilinear attention pooling and "\n            "attention-guided crop/drop; not a byte-identical reproduction"\n        ),\n        "fixed_conditions": {\n            "data_sha256": context["data_hash"],\n            "split": "same original Validation 500 images",\n            "train_variant": "augmented",\n            "backbone": "EfficientNet-B0",\n            "input_size": 256,\n            "loss": "categorical_crossentropy",\n            "seed": 42,\n            "stage1_epochs": context["config"]["epochs1"],\n            "stage2_epochs": context["config"]["epochs2"],\n            "test_access": False,\n        },\n        "changed_condition": (\n            "global average pooling and random stored augmentation versus WS-DAN-inspired "\n            "bilinear attention pooling plus online attention crop/drop"\n        ),\n        "baseline_retrained": False,\n        "baseline": BASELINE,\n        "wsdan": record["validation"],\n        "delta_wsdan_minus_baseline": {\n            "validation_accuracy": record["validation"]["accuracy"]\n            - BASELINE["validation_accuracy"],\n            "validation_macro_f1": record["validation"]["macro_f1"]\n            - BASELINE["validation_macro_f1"],\n        },\n        "screening_only": True,\n        "test_evaluated": False,\n    }\n    engine.write_json(output / "wsdan_comparison_summary.json", summary)\n    history = record["history"]\n    epochs = np.arange(1, len(history["accuracy"]) + 1)\n    fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n    axes[0].plot(epochs, history["accuracy"], label="Train")\n    axes[0].plot(epochs, history["val_accuracy"], label="Validation")\n    axes[0].axvline(record["stage_boundary"] + 0.5, color="gray", linestyle="--")\n    axes[0].set(title="WS-DAN Accuracy", xlabel="Epoch", ylabel="Accuracy")\n    axes[1].plot(epochs, history["loss"], label="Train")\n    axes[1].plot(epochs, history["val_loss"], label="Validation")\n    axes[1].axvline(record["stage_boundary"] + 0.5, color="gray", linestyle="--")\n    axes[1].set(title="WS-DAN Loss", xlabel="Epoch", ylabel="Loss")\n    for axis in axes:\n        axis.grid(alpha=0.25)\n        axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / "wsdan_training_curves.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    archive = flow.archive_results(context)\n    print("Web Skin WS-DAN Validation 선별 완료:", output)\n    print("기존 B0·256·CE는 다시 학습하지 않았고 Test도 평가하지 않았습니다.")\n    return summary, archive\n', 'web_skin_paper_suite': '"""Three validation-only paper experiments for the Web Skin classifier.\n\nEach method is trained independently.  Historical B0/256/CE validation metrics\nare a read-only reference and Test is deliberately never loaded.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import common_engine as engine\nfrom mediflow_datasets import common_workflow as flow\nfrom mediflow_datasets import web_skin_wsdan as wsdan\n\nPROTOCOL = "web_skin_three_papers_v2"\nEXPERIMENTS = [\n    wsdan.TRIAL_ID,\n    "pmg_b0_256_ce_seed_42",\n    "mixstyle_b0_256_ce_seed_42",\n]\nPAPERS = {\n    "wsdan": "https://arxiv.org/abs/1901.09891",\n    "pmg": "https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123650154.pdf",\n    "mixstyle": "https://openreview.net/forum?id=6xHJ37MVxxp",\n}\n\n\ndef validate(context):\n    config = context["config"]\n    if config["domain"] != "web_skin" or config["mode"] != "web_skin_paper_suite":\n        raise ValueError("이 코드는 Web Skin 3개 논문 실험 전용입니다.")\n    if config["train_variant"] != "augmented":\n        raise ValueError("기존 기준선과 같은 augmented Train을 사용해야 합니다.")\n    if config["seed"] != 42 or config["seeds"] != [42]:\n        raise ValueError("논문 방법 선별 seed는 42로 고정합니다.")\n    if config.get("experiments") != EXPERIMENTS:\n        raise ValueError("WS-DAN, PMG, MixStyle 세 조건이 필요합니다.")\n    if context["classes"] != wsdan.EXPECTED_CLASSES:\n        raise ValueError("Web Skin 클래스 순서가 기존 모델 계약과 다릅니다.")\n    if context["data_hash"] != wsdan.BASELINE["data_sha256"]:\n        raise ValueError("저장된 기준선과 데이터 SHA-256이 다릅니다.")\n    if config.get("pmg_jigsaw_grids") != [8, 4, 2]:\n        raise ValueError("PMG jigsaw 순서는 원 논문의 8, 4, 2로 고정합니다.")\n    for key in ("mixstyle_alpha", "mixstyle_probability"):\n        value = config.get(key)\n        if not isinstance(value, (float, int)) or not 0 < value <= 1:\n            raise ValueError(key + "는 0보다 크고 1 이하여야 합니다.")\n\n\ndef _trial_signature(context, spec):\n    return hashlib.sha256(\n        (context["signature"] + json.dumps(spec, sort_keys=True)).encode()\n    ).hexdigest()\n\n\ndef _attempt(context, trial_id):\n    directory = (\n        Path(context["output"]) / trial_id / ("attempt_" + uuid.uuid4().hex[:12])\n    )\n    directory.mkdir(parents=True, exist_ok=False)\n    return directory\n\n\ndef _cache_or_none(context, trial_id, signature):\n    return engine.cached_record(context["output"], trial_id, signature)\n\n\ndef _run_wsdan(context):\n    return wsdan.run(context)\n\n\ndef _jigsaw(images, grid):\n    shape = tf.shape(images)\n    batch, height, width, channels = shape[0], shape[1], shape[2], shape[3]\n    patch_height, patch_width = height // grid, width // grid\n    patches = tf.reshape(\n        images,\n        (batch, grid, patch_height, grid, patch_width, channels),\n    )\n    patches = tf.transpose(patches, (0, 1, 3, 2, 4, 5))\n    patches = tf.reshape(\n        patches, (batch, grid * grid, patch_height, patch_width, channels)\n    )\n    patches = tf.gather(patches, tf.random.shuffle(tf.range(grid * grid)), axis=1)\n    patches = tf.reshape(\n        patches, (batch, grid, grid, patch_height, patch_width, channels)\n    )\n    patches = tf.transpose(patches, (0, 1, 3, 2, 4, 5))\n    return tf.reshape(patches, (batch, height, width, channels))\n\n\ndef build_pmg_model(class_count, size=256, weights="imagenet", backbone_name="B0"):\n    builders = {\n        "B0": keras.applications.EfficientNetB0,\n        "B1": keras.applications.EfficientNetB1,\n    }\n    if backbone_name not in builders:\n        raise ValueError("PMG backbone은 B0 또는 B1이어야 합니다.")\n    backbone = builders[backbone_name](\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    feature_model = keras.Model(\n        backbone.input,\n        [\n            backbone.get_layer("block3a_expand_activation").output,\n            backbone.get_layer("block5a_expand_activation").output,\n            backbone.get_layer("top_activation").output,\n        ],\n        name="pmg_backbone",\n    )\n    feature_model.trainable = False\n    inputs = keras.Input((size, size, 3), name="image")\n    features = feature_model(inputs, training=False)\n    embeddings, logits = [], []\n    for index, feature in enumerate(features, 1):\n        value = keras.layers.Conv2D(\n            256, 1, activation="relu", name=f"pmg_branch{index}_reduce"\n        )(feature)\n        value = keras.layers.Conv2D(\n            512, 3, padding="same", activation="relu", name=f"pmg_branch{index}_conv"\n        )(value)\n        value = keras.layers.GlobalAveragePooling2D(\n            name=f"pmg_branch{index}_pool"\n        )(value)\n        value = keras.layers.Dense(\n            256, activation="elu", name=f"pmg_branch{index}_embedding"\n        )(value)\n        embeddings.append(value)\n        logits.append(\n            keras.layers.Dense(class_count, name=f"pmg_branch{index}_logits")(value)\n        )\n    combined = keras.layers.Concatenate(name="pmg_concat")(embeddings)\n    combined = keras.layers.Dense(512, activation="elu", name="pmg_fusion")(combined)\n    logits.append(keras.layers.Dense(class_count, name="pmg_concat_logits")(combined))\n    return keras.Model(inputs, logits, name="web_skin_pmg_" + backbone_name.lower())\n\n\ndef _configure_pmg_partial(model):\n    backbone = model.get_layer("pmg_backbone")\n    backbone.trainable = True\n    for index, layer in enumerate(backbone.layers):\n        layer.trainable = index >= len(backbone.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    return [layer.name for layer in backbone.layers if layer.trainable]\n\n\ndef _pmg_predictor(model):\n    total = keras.layers.Add(name="pmg_logit_sum")(model.outputs)\n    probabilities = keras.layers.Activation("softmax", name="predictions")(total)\n    return keras.Model(model.input, probabilities, name="web_skin_pmg_inference")\n\n\ndef _fit_pmg_stage(model, train, val, directory, name, epochs, learning_rate):\n    optimizer = keras.optimizers.Adam(learning_rate)\n    optimizer.build(model.trainable_variables)\n    loss_function = keras.losses.CategoricalCrossentropy(from_logits=True)\n    history = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}\n    best_score, best = -1.0, directory / f"{name}_best.keras"\n\n    @tf.function\n    def branch_step(images, labels, branch, grid):\n        with tf.GradientTape() as tape:\n            outputs = model(_jigsaw(images, grid), training=True)\n            loss = loss_function(labels, outputs[branch])\n        gradients = tape.gradient(loss, model.trainable_variables)\n        optimizer.apply_gradients(\n            [\n                (g, v)\n                for g, v in zip(gradients, model.trainable_variables, strict=True)\n                if g is not None\n            ]\n        )\n        return loss\n\n    @tf.function\n    def fusion_step(images, labels):\n        with tf.GradientTape() as tape:\n            outputs = model(images, training=True)\n            loss = 2.0 * loss_function(labels, outputs[3])\n        gradients = tape.gradient(loss, model.trainable_variables)\n        optimizer.apply_gradients(\n            [\n                (g, v)\n                for g, v in zip(gradients, model.trainable_variables, strict=True)\n                if g is not None\n            ]\n        )\n        return loss, tf.nn.softmax(tf.add_n(outputs), axis=-1)\n\n    predictor = _pmg_predictor(model)\n    for epoch in range(epochs):\n        train_loss = keras.metrics.Mean()\n        train_accuracy = keras.metrics.CategoricalAccuracy()\n        for images, labels in train:\n            losses = [\n                branch_step(images, labels, 0, 8),\n                branch_step(images, labels, 1, 4),\n                branch_step(images, labels, 2, 2),\n            ]\n            fusion_loss, predictions = fusion_step(images, labels)\n            train_loss.update_state(tf.add_n(losses) + fusion_loss)\n            train_accuracy.update_state(labels, predictions)\n        val_loss = keras.metrics.Mean()\n        val_accuracy = keras.metrics.CategoricalAccuracy()\n        for images, labels in val:\n            outputs = model(images, training=False)\n            val_loss.update_state(loss_function(labels, outputs[3]))\n            val_accuracy.update_state(labels, predictor(images, training=False))\n        values = {\n            "accuracy": float(train_accuracy.result()),\n            "loss": float(train_loss.result()),\n            "val_accuracy": float(val_accuracy.result()),\n            "val_loss": float(val_loss.result()),\n        }\n        for key, value in values.items():\n            history[key].append(value)\n        engine.write_json(directory / f"{name}_history.json", history)\n        print(name, "Epoch", epoch + 1, "/", epochs, values)\n        if values["val_accuracy"] > best_score:\n            best_score = values["val_accuracy"]\n            model.save(best)\n    model.save(directory / f"{name}_last.keras")\n    return history, best\n\n\ndef _run_pmg(context):\n    config, trial_id = context["config"], EXPERIMENTS[1]\n    spec = {\n        "id": trial_id,\n        "method": "efficientnet_b0_pmg_adaptation",\n        "paper": PAPERS["pmg"],\n        "jigsaw_grids": [8, 4, 2],\n        "input_size": 256,\n        "seed": 42,\n        "test_evaluated": False,\n    }\n    signature = _trial_signature(context, spec)\n    cached = _cache_or_none(context, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(42)\n    directory = _attempt(context, trial_id)\n    engine.write_json(directory / "spec.json", spec)\n    train, _ = flow.factory(context, "augmented", seed=42)("train", 256, True)\n    val, paths = flow.factory(context, "augmented", seed=42)("val", 256, False)\n    started = time.monotonic()\n    model = build_pmg_model(len(context["classes"]))\n    h1, best1 = _fit_pmg_stage(\n        model, train, val, directory, "stage1", config["epochs1"], 1e-4\n    )\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = _configure_pmg_partial(model)\n    h2, best2 = _fit_pmg_stage(\n        model, train, val, directory, "stage2", config["epochs2"], 1e-5\n    )\n    selected_stage = (\n        "stage2"\n        if engine.checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"]))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(selected, compile=False)\n    predictor = _pmg_predictor(model)\n    metrics = engine.evaluate_to_files(predictor, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]),\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n        "inference": "sum of three branch logits and fused logits on one original image",\n        "test_evaluated": False,\n    }\n    engine.finish_record(directory, record)\n    return record\n\n\n@keras.saving.register_keras_serializable(package="MediFlow")\nclass MixStyle(keras.layers.Layer):\n    def __init__(self, probability=0.5, alpha=0.1, **kwargs):\n        super().__init__(**kwargs)\n        self.probability = probability\n        self.alpha = alpha\n\n    def call(self, inputs, training=None):\n        if training is not True:\n            return inputs\n        mean = tf.reduce_mean(inputs, axis=(1, 2), keepdims=True)\n        variance = tf.reduce_mean(tf.square(inputs - mean), axis=(1, 2), keepdims=True)\n        deviation = tf.sqrt(variance + 1e-6)\n        normalized = (inputs - tf.stop_gradient(mean)) / tf.stop_gradient(deviation)\n        permutation = tf.random.shuffle(tf.range(tf.shape(inputs)[0]))\n        mixed_mean = tf.gather(mean, permutation)\n        mixed_deviation = tf.gather(deviation, permutation)\n        first = tf.random.gamma((tf.shape(inputs)[0], 1, 1, 1), self.alpha)\n        second = tf.random.gamma((tf.shape(inputs)[0], 1, 1, 1), self.alpha)\n        weight = first / (first + second + 1e-6)\n        target_mean = weight * mean + (1.0 - weight) * mixed_mean\n        target_deviation = weight * deviation + (1.0 - weight) * mixed_deviation\n        styled = normalized * tf.stop_gradient(target_deviation) + tf.stop_gradient(\n            target_mean\n        )\n        return tf.cond(\n            tf.random.uniform(()) < self.probability, lambda: styled, lambda: inputs\n        )\n\n    def get_config(self):\n        return {\n            **super().get_config(),\n            "probability": self.probability,\n            "alpha": self.alpha,\n        }\n\n\ndef build_mixstyle_model(\n    class_count, size=256, probability=0.5, alpha=0.1, weights="imagenet"\n):\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    split = backbone.get_layer("block2b_add").output\n    early = keras.Model(backbone.input, split, name="mixstyle_early")\n    late = keras.Model(split, backbone.output, name="mixstyle_late")\n    early.trainable = False\n    late.trainable = False\n    inputs = keras.Input((size, size, 3), name="image")\n    features = early(inputs, training=False)\n    features = MixStyle(probability, alpha, name="mixstyle")(features)\n    features = late(features, training=False)\n    features = keras.layers.GlobalAveragePooling2D()(features)\n    features = keras.layers.Dropout(0.3)(features)\n    outputs = keras.layers.Dense(class_count, activation="softmax", name="predictions")(\n        features\n    )\n    model = keras.Model(inputs, outputs, name="web_skin_mixstyle_b0")\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-4),\n        loss=keras.losses.CategoricalCrossentropy(),\n        metrics=["accuracy"],\n        # MixStyle samples Beta weights through RandomGamma. Keras 3 may enable\n        # XLA automatically on GPU, but TensorFlow has no XLA GPU kernel for\n        # RandomGamma. Keep the normal compiled TensorFlow graph for this model.\n        jit_compile=False,\n    )\n    return model\n\n\ndef _configure_mixstyle_partial(model):\n    late = model.get_layer("mixstyle_late")\n    late.trainable = True\n    for index, layer in enumerate(late.layers):\n        layer.trainable = index >= len(late.layers) - 30 and not isinstance(\n            layer, keras.layers.BatchNormalization\n        )\n    model.compile(\n        optimizer=keras.optimizers.Adam(1e-5),\n        loss=keras.losses.CategoricalCrossentropy(),\n        metrics=["accuracy"],\n        jit_compile=False,\n    )\n    return [layer.name for layer in late.layers if layer.trainable]\n\n\ndef _run_mixstyle(context):\n    config, trial_id = context["config"], EXPERIMENTS[2]\n    spec = {\n        "id": trial_id,\n        "method": "mixstyle_after_efficientnet_block2b",\n        "paper": PAPERS["mixstyle"],\n        "alpha": float(config["mixstyle_alpha"]),\n        "probability": float(config["mixstyle_probability"]),\n        "input_size": 256,\n        "seed": 42,\n        "test_evaluated": False,\n    }\n    signature = _trial_signature(context, spec)\n    cached = _cache_or_none(context, trial_id, signature)\n    if cached:\n        return cached\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(42)\n    directory = _attempt(context, trial_id)\n    engine.write_json(directory / "spec.json", spec)\n    train, _ = flow.factory(context, "augmented", seed=42)("train", 256, True)\n    val, paths = flow.factory(context, "augmented", seed=42)("val", 256, False)\n    started = time.monotonic()\n    model = build_mixstyle_model(\n        len(context["classes"]),\n        probability=config["mixstyle_probability"],\n        alpha=config["mixstyle_alpha"],\n    )\n    h1, best1 = engine.fit_stage(\n        model, train, val, directory, "stage1", config["epochs1"]\n    )\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(best1, compile=False)\n    trainable = _configure_mixstyle_partial(model)\n    h2, best2 = engine.fit_stage(\n        model, train, val, directory, "stage2", config["epochs2"]\n    )\n    selected_stage = (\n        "stage2"\n        if engine.checkpoint_choice(max(h1["val_accuracy"]), max(h2["val_accuracy"]))\n        else "stage1"\n    )\n    selected = best2 if selected_stage == "stage2" else best1\n    del model\n    keras.backend.clear_session()\n    model = keras.models.load_model(selected, compile=False)\n    metrics = engine.evaluate_to_files(model, val, paths, directory, "validation")\n    record = {\n        "id": trial_id,\n        "spec": spec,\n        "signature": signature,\n        "attempt": directory.name,\n        "selected_model": selected.name,\n        "selected_stage": selected_stage,\n        "validation": metrics,\n        "stage1_best_val": max(h1["val_accuracy"]),\n        "stage2_best_val": max(h2["val_accuracy"]),\n        "training_seconds": time.monotonic() - started,\n        "parameters": model.count_params(),\n        "model_bytes": selected.stat().st_size,\n        "trainable_backbone_layers": trainable,\n        "history": {key: h1[key] + h2[key] for key in h1},\n        "stage_boundary": len(h1["accuracy"]),\n        "inference": "MixStyle disabled automatically; one original image",\n        "test_evaluated": False,\n    }\n    engine.finish_record(directory, record)\n    return record\n\n\ndef run(context):\n    validate(context)\n    records = []\n    runners = (_run_wsdan, _run_pmg, _run_mixstyle)\n    try:\n        for runner in runners:\n            record = runner(context)\n            records.append(record)\n            engine.write_json(\n                Path(context["output"]) / "progress.json",\n                {"completed": [item["id"] for item in records]},\n            )\n        engine.write_json(\n            Path(context["output"]) / "all_validation_results.json", records\n        )\n        return records\n    except Exception as exc:\n        engine.write_json(\n            Path(context["output"]) / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [item["id"] for item in records],\n                "resume_dir": str(context["output"]),\n            },\n        )\n        print("중단 전 완료 실험은 보존됐습니다. RESUME_DIR:", context["output"])\n        raise\n\n\ndef _confusion(axis, matrix, title):\n    matrix = np.asarray(matrix, dtype=int)\n    image = axis.imshow(matrix, cmap="Blues")\n    for row in range(len(matrix)):\n        for column in range(len(matrix)):\n            axis.text(\n                column,\n                row,\n                str(matrix[row, column]),\n                ha="center",\n                va="center",\n                color="white" if matrix[row, column] > matrix.max() / 2 else "black",\n            )\n    axis.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(matrix)),\n        yticks=range(len(matrix)),\n        xticklabels=[f"C{i}" for i in range(len(matrix))],\n        yticklabels=[f"C{i}" for i in range(len(matrix))],\n    )\n    return image\n\n\ndef summarize(context, records):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    validate(context)\n    if not records:\n        raise ValueError("완료된 Web Skin 논문 실험이 없습니다.")\n    record_ids = [record["id"] for record in records]\n    if len(set(record_ids)) != len(record_ids) or any(\n        trial_id not in EXPERIMENTS for trial_id in record_ids\n    ):\n        raise ValueError("완료 실험 목록에 중복 또는 알 수 없는 항목이 있습니다.")\n    output = Path(context["output"])\n    baseline = {\n        "experiment": "saved_b0_256_ce_baseline",\n        "validation_accuracy": wsdan.BASELINE["validation_accuracy"],\n        "validation_macro_f1": wsdan.BASELINE["validation_macro_f1"],\n        "trained_in_this_run": False,\n    }\n    rows = [baseline]\n    for record in records:\n        rows.append(\n            {\n                "experiment": record["id"],\n                "validation_accuracy": record["validation"]["accuracy"],\n                "validation_macro_f1": record["validation"]["macro_f1"],\n                "trained_in_this_run": True,\n            }\n        )\n    table = pd.DataFrame(rows)\n    table.to_csv(output / "paper_method_comparison.csv", index=False, encoding="utf-8-sig")\n    print(table.to_string(index=False))\n\n    label_by_id = {\n        EXPERIMENTS[0]: "WS-DAN",\n        EXPERIMENTS[1]: "PMG",\n        EXPERIMENTS[2]: "MixStyle",\n    }\n    labels = ["Baseline"] + [label_by_id[trial_id] for trial_id in record_ids]\n    x = np.arange(len(labels))\n    fig, axis = plt.subplots(figsize=(15, 6))\n    axis.bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axis.bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    for index, value in enumerate(table.validation_accuracy):\n        axis.text(index - 0.2, value + 0.005, f"{value:.4f}", ha="center")\n    for index, value in enumerate(table.validation_macro_f1):\n        axis.text(index + 0.2, value + 0.005, f"{value:.4f}", ha="center")\n    axis.set_xticks(x, labels)\n    axis.set_ylim(0, 1)\n    axis.set_title("Web Skin Paper-based Method Comparison")\n    axis.grid(axis="y", alpha=0.25)\n    axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / "paper_method_performance_dashboard.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    fig, axes = plt.subplots(\n        1, len(records), figsize=(5 * len(records), 5), squeeze=False\n    )\n    for axis, record, label in zip(axes.flat, records, labels[1:], strict=True):\n        _confusion(axis, record["validation"]["confusion_matrix"], label)\n    fig.suptitle("Web Skin Validation Confusion Matrices (C0-C4)")\n    fig.tight_layout()\n    fig.savefig(output / "paper_method_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    fig, axes = plt.subplots(\n        1, len(records), figsize=(6 * len(records), 5), squeeze=False\n    )\n    for axis, record, label in zip(axes.flat, records, labels[1:], strict=True):\n        history = record["history"]\n        epochs = np.arange(1, len(history["accuracy"]) + 1)\n        axis.plot(epochs, history["accuracy"], label="Train accuracy")\n        axis.plot(epochs, history["val_accuracy"], label="Validation accuracy")\n        if record["stage_boundary"] < len(epochs):\n            axis.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n        axis.set(title=label, xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))\n        axis.grid(alpha=0.25)\n        axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / "paper_method_training_curves.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    class_rows = [wsdan.BASELINE["class_f1"]] + [\n        record["validation"]["class_f1"] for record in records\n    ]\n    frame = pd.DataFrame(class_rows, index=labels, columns=context["classes"])\n    frame.to_csv(output / "paper_method_class_f1.csv", encoding="utf-8-sig")\n    summary = {\n        "protocol": PROTOCOL,\n        "data_sha256": context["data_hash"],\n        "classes": context["classes"],\n        "baseline_retrained": False,\n        "test_evaluated": False,\n        "completed_experiments": record_ids,\n        "omitted_experiments": [\n            trial_id for trial_id in EXPERIMENTS if trial_id not in record_ids\n        ],\n        "papers": PAPERS,\n        "comparison": rows,\n        "winner_by_validation_macro_f1": table.iloc[\n            int(table.validation_macro_f1.argmax())\n        ].to_dict(),\n    }\n    engine.write_json(output / "paper_method_summary.json", summary)\n    archive = flow.archive_results(context)\n    return summary, archive\n', 'web_skin_pmg_b1_384': '"""Single Web Skin PMG B1/384 validation experiment; Test remains unopened."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\n\nfrom mediflow_datasets import common_engine as engine\nfrom mediflow_datasets import common_workflow as flow\nfrom mediflow_datasets import web_skin_paper_suite as pmg\nfrom mediflow_datasets import web_skin_wsdan as contract\n\nPROTOCOL = "web_skin_pmg_b1_384_screen_v1"\nTRIAL_ID = "pmg_b1_384_ce_seed_42"\nEXPERIMENTS = [TRIAL_ID]\nPAPERS = {\n    "pmg": "https://www.ecva.net/papers/eccv_2020/papers_ECCV/papers/123650154.pdf",\n    "efficientnet": "https://proceedings.mlr.press/v97/tan19a.html",\n}\nBASELINE = {\n    "id": "pmg_b0_256_ce_seed_42",\n    "validation_accuracy": 0.85,\n    "validation_macro_f1": 0.8473279632397033,\n    "validation_count": 500,\n    "class_f1": [\n        0.8252427184466019,\n        0.7835051546391754,\n        0.7634408602150538,\n        0.9615384615384615,\n        0.9029126213592233,\n    ],\n    "confusion_matrix": [\n        [85, 5, 4, 3, 3],\n        [7, 76, 8, 1, 8],\n        [12, 12, 71, 3, 2],\n        [0, 0, 0, 100, 0],\n        [2, 1, 3, 1, 93],\n    ],\n    "data_sha256": contract.BASELINE["data_sha256"],\n    "source": (\n        "web_skin_paper_suite_20260922_124758_717b465e/"\n        "pmg_b0_256_ce_seed_42/attempt_a8364775fcb8/validation_metrics.json"\n    ),\n    "test_evaluated": False,\n}\n\n\ndef validate(context):\n    config = context["config"]\n    if config["domain"] != "web_skin" or config["mode"] != "web_skin_pmg_b1_384":\n        raise ValueError("이 코드는 Web Skin PMG·B1·384 선별 실험 전용입니다.")\n    if config["train_variant"] != "augmented":\n        raise ValueError("기존 PMG와 같은 augmented Train을 사용해야 합니다.")\n    if config["seed"] != 42 or config["seeds"] != [42]:\n        raise ValueError("선별 seed는 42로 고정합니다.")\n    if config.get("experiments") != EXPERIMENTS:\n        raise ValueError("PMG·B1·384 단일 조건만 실행해야 합니다.")\n    if config.get("pmg_jigsaw_grids") != [8, 4, 2]:\n        raise ValueError("PMG jigsaw 순서는 8, 4, 2로 고정합니다.")\n    if config["batch_size"] != 16:\n        raise ValueError("384 입력의 Colab GPU 메모리를 위해 batch size는 16입니다.")\n    if config["epochs1"] != 15 or config["epochs2"] != 10:\n        raise ValueError("기존 PMG와 같이 Stage 1=15, Stage 2=10으로 고정합니다.")\n    if context["classes"] != contract.EXPECTED_CLASSES:\n        raise ValueError("Web Skin 클래스 순서가 기존 모델 계약과 다릅니다.")\n    if context["data_hash"] != BASELINE["data_sha256"]:\n        raise ValueError("기존 PMG와 데이터 SHA-256이 다릅니다.")\n\n\ndef _signature(context, spec):\n    return hashlib.sha256(\n        (context["signature"] + json.dumps(spec, sort_keys=True)).encode()\n    ).hexdigest()\n\n\ndef _attempt(context):\n    directory = (\n        Path(context["output"]) / TRIAL_ID / ("attempt_" + uuid.uuid4().hex[:12])\n    )\n    directory.mkdir(parents=True, exist_ok=False)\n    return directory\n\n\ndef run(context):\n    validate(context)\n    config = context["config"]\n    spec = {\n        "id": TRIAL_ID,\n        "method": "efficientnet_b1_384_pmg_adaptation",\n        "papers": PAPERS,\n        "backbone": "B1",\n        "input_size": 384,\n        "loss": "ce",\n        "jigsaw_grids": [8, 4, 2],\n        "batch_size": 16,\n        "epochs1": 15,\n        "epochs2": 10,\n        "seed": 42,\n        "selection_split": "validation",\n        "test_evaluated": False,\n    }\n    signature = _signature(context, spec)\n    cached = engine.cached_record(context["output"], TRIAL_ID, signature)\n    if cached:\n        return cached\n\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(42)\n    directory = _attempt(context)\n    engine.write_json(directory / "spec.json", spec)\n    engine.write_json(Path(context["output"]) / "baseline_reference.json", BASELINE)\n    train, _ = flow.factory(context, "augmented", seed=42)("train", 384, True)\n    val, paths = flow.factory(context, "augmented", seed=42)("val", 384, False)\n    started = time.monotonic()\n    try:\n        model = pmg.build_pmg_model(\n            len(context["classes"]), size=384, backbone_name="B1"\n        )\n        h1, best1 = pmg._fit_pmg_stage(\n            model, train, val, directory, "stage1", config["epochs1"], 1e-4\n        )\n        del model\n        keras.backend.clear_session()\n        model = keras.models.load_model(best1, compile=False)\n        trainable = pmg._configure_pmg_partial(model)\n        h2, best2 = pmg._fit_pmg_stage(\n            model, train, val, directory, "stage2", config["epochs2"], 1e-5\n        )\n        selected_stage = (\n            "stage2"\n            if engine.checkpoint_choice(\n                max(h1["val_accuracy"]), max(h2["val_accuracy"])\n            )\n            else "stage1"\n        )\n        selected = best2 if selected_stage == "stage2" else best1\n        del model\n        keras.backend.clear_session()\n        model = keras.models.load_model(selected, compile=False)\n        predictor = pmg._pmg_predictor(model)\n        metrics = engine.evaluate_to_files(\n            predictor, val, paths, directory, "validation"\n        )\n        record = {\n            "id": TRIAL_ID,\n            "spec": spec,\n            "signature": signature,\n            "attempt": directory.name,\n            "selected_model": selected.name,\n            "selected_stage": selected_stage,\n            "validation": metrics,\n            "stage1_best_val": max(h1["val_accuracy"]),\n            "stage2_best_val": max(h2["val_accuracy"]),\n            "training_seconds": time.monotonic() - started,\n            "parameters": model.count_params(),\n            "model_bytes": selected.stat().st_size,\n            "trainable_backbone_layers": trainable,\n            "history": {key: h1[key] + h2[key] for key in h1},\n            "stage_boundary": len(h1["accuracy"]),\n            "inference": (\n                "one 384x384 image; sum of three branch logits and fused logits"\n            ),\n            "test_evaluated": False,\n        }\n        engine.finish_record(directory, record)\n        return record\n    except Exception as exc:\n        engine.write_json(\n            Path(context["output"]) / ("failure_" + uuid.uuid4().hex[:8] + ".json"),\n            {\n                "error": repr(exc),\n                "completed": [],\n                "resume_dir": str(context["output"]),\n                "attempt": directory.name,\n            },\n        )\n        print("실패 attempt는 보존됐습니다. RESUME_DIR:", context["output"])\n        raise\n\n\ndef _draw_confusion(axis, matrix, title):\n    matrix = np.asarray(matrix, dtype=int)\n    image = axis.imshow(matrix, cmap="Blues")\n    for row in range(len(matrix)):\n        for column in range(len(matrix)):\n            axis.text(\n                column,\n                row,\n                str(matrix[row, column]),\n                ha="center",\n                va="center",\n                color="white" if matrix[row, column] > matrix.max() / 2 else "black",\n            )\n    axis.set(\n        title=title,\n        xlabel="Predicted",\n        ylabel="True",\n        xticks=range(len(matrix)),\n        yticks=range(len(matrix)),\n        xticklabels=[f"C{i}" for i in range(len(matrix))],\n        yticklabels=[f"C{i}" for i in range(len(matrix))],\n    )\n    return image\n\n\ndef summarize(context, record):\n    import matplotlib.pyplot as plt\n    import pandas as pd\n\n    validate(context)\n    if record["id"] != TRIAL_ID or record.get("test_evaluated") is not False:\n        raise ValueError("PMG·B1·384 Validation 완료 기록이 아닙니다.")\n    output = Path(context["output"])\n    rows = [\n        {\n            "experiment": BASELINE["id"],\n            "validation_accuracy": BASELINE["validation_accuracy"],\n            "validation_macro_f1": BASELINE["validation_macro_f1"],\n            "trained_in_this_run": False,\n        },\n        {\n            "experiment": record["id"],\n            "validation_accuracy": record["validation"]["accuracy"],\n            "validation_macro_f1": record["validation"]["macro_f1"],\n            "trained_in_this_run": True,\n        },\n    ]\n    table = pd.DataFrame(rows)\n    table.to_csv(output / "pmg_scale_comparison.csv", index=False, encoding="utf-8-sig")\n\n    labels = ["PMG B0·256", "PMG B1·384"]\n    x = np.arange(2)\n    fig, axis = plt.subplots(figsize=(10, 6))\n    axis.bar(x - 0.2, table.validation_accuracy, 0.4, label="Validation Accuracy")\n    axis.bar(x + 0.2, table.validation_macro_f1, 0.4, label="Validation Macro F1")\n    for index, value in enumerate(table.validation_accuracy):\n        axis.text(index - 0.2, value + 0.005, f"{value:.4f}", ha="center")\n    for index, value in enumerate(table.validation_macro_f1):\n        axis.text(index + 0.2, value + 0.005, f"{value:.4f}", ha="center")\n    axis.set_xticks(x, labels)\n    axis.set_ylim(0, 1)\n    axis.set_title("Web Skin PMG Scale Comparison")\n    axis.grid(axis="y", alpha=0.25)\n    axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / "pmg_scale_performance.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(11, 5))\n    _draw_confusion(axes[0], BASELINE["confusion_matrix"], labels[0])\n    _draw_confusion(axes[1], record["validation"]["confusion_matrix"], labels[1])\n    fig.suptitle("Web Skin PMG Validation Confusion Matrices (C0-C4)")\n    fig.tight_layout()\n    fig.savefig(output / "pmg_scale_confusion_matrices.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    history = record["history"]\n    epochs = np.arange(1, len(history["accuracy"]) + 1)\n    fig, axes = plt.subplots(1, 2, figsize=(13, 5))\n    axes[0].plot(epochs, history["accuracy"], label="Train")\n    axes[0].plot(epochs, history["val_accuracy"], label="Validation")\n    axes[1].plot(epochs, history["loss"], label="Train")\n    axes[1].plot(epochs, history["val_loss"], label="Validation")\n    for axis, title, ylabel in zip(\n        axes, ("Accuracy", "Loss"), ("Accuracy", "Loss"), strict=True\n    ):\n        axis.axvline(record["stage_boundary"] + 0.5, ls="--", color="gray")\n        axis.set(title=title, xlabel="Epoch", ylabel=ylabel)\n        axis.grid(alpha=0.25)\n        axis.legend()\n    fig.tight_layout()\n    fig.savefig(output / "pmg_b1_384_training_curves.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n\n    class_frame = pd.DataFrame(\n        [BASELINE["class_f1"], record["validation"]["class_f1"]],\n        index=labels,\n        columns=context["classes"],\n    )\n    class_frame.to_csv(output / "pmg_scale_class_f1.csv", encoding="utf-8-sig")\n    winner = max(rows, key=lambda item: item["validation_macro_f1"])\n    summary = {\n        "protocol": PROTOCOL,\n        "data_sha256": context["data_hash"],\n        "classes": context["classes"],\n        "baseline_retrained": False,\n        "completed_experiments": [TRIAL_ID],\n        "comparison": rows,\n        "winner_by_validation_macro_f1": winner,\n        "test_evaluated": False,\n        "necessary_accompanying_change": (\n            "Batch size 16 is used for B1/384 GPU memory; the earlier B0/256 PMG used 32."\n        ),\n    }\n    engine.write_json(output / "pmg_scale_summary.json", summary)\n    archive = flow.archive_results(context)\n    return summary, archive\n'}
BUILD_COMMIT = '5997df7e2d4e3ea2b26417f4725467140e9d6f64'
PROFILES = {'web_skin': ['건선', '아토피', '여드름', '정상', '주사']}
package = types.ModuleType('mediflow_datasets')
package.__path__ = []
sys.modules['mediflow_datasets'] = package
for name, source in SOURCES.items():
    module = types.ModuleType('mediflow_datasets.' + name)
    sys.modules[module.__name__] = module
    exec(compile(source, name + '.py', 'exec'), module.__dict__)
from mediflow_datasets.common_workflow import prepare
from mediflow_datasets.web_skin_pmg_b1_384 import run, summarize


## 4. 데이터 준비와 실행 폴더 생성

데이터 SHA-256과 클래스 순서를 확인합니다. 중단 후 다시 실행할 때 출력된 결과 폴더를
`RESUME_DIR`에 입력합니다. 완료 모델은 검증 후 재사용하고, 완료되지 않은 attempt는 보존한
채 새 attempt에서 다시 시작합니다.


In [ ]:
config = dict(
    domain=DOMAIN, project_root=PROJECT_ROOT, data_zip=DATA_ZIP,
    audit_dir=AUDIT_DIR, expected_data_sha256=EXPECTED_DATA_SHA256,
    resume_dir=RESUME_DIR, mode=MODE, seed=SEED, seeds=SEEDS,
    batch_size=BATCH_SIZE, epochs1=STAGE1_EPOCHS, epochs2=STAGE2_EPOCHS,
    extension_epochs=EXTENSION_EPOCHS, train_variant=TRAIN_VARIANT,
    experiments=RUN_EXPERIMENTS, pmg_jigsaw_grids=PMG_JIGSAW_GRIDS,
)
context = prepare(config, PROFILES, SOURCES, BUILD_COMMIT)
print('결과 폴더 / 중단 시 RESUME_DIR:', context['output'])
print('데이터 SHA-256:', context['data_hash'])
print('클래스 순서:', context['classes'])


## 5. PMG·B1·384 한 모델 학습

Stage 1은 분류부 중심으로 15 epoch, Stage 2는 EfficientNet-B1 후반부를 부분 미세조정하여
10 epoch 학습합니다. 모델은 Validation Accuracy가 가장 높았던 checkpoint를 저장합니다.


In [ ]:
record = run(context)
record['validation']


## 6. 기존 PMG와 Validation 비교 자료 생성

기존 PMG·B0·256과 새 PMG·B1·384의 Accuracy, Macro F1, 클래스별 F1, 혼동행렬과 학습곡선을
저장합니다. Macro F1이 더 높은 모델을 최종 Test 후보로 표시하지만 Test 자체는 실행하지
않습니다. `.keras` 모델은 Drive 실행 폴더에 남고 결과 ZIP에서는 제외됩니다.


In [ ]:
summary, report_zip = summarize(context, record)
summary
